# NTNU Workshop — Kelvin lattice: X-ray projections → neural_xray reconstruction

**Default path (one-click):** install → download data → train → visualise.

**Run cells 1 → 2 → 5 → 6 → 7.** Skip cells 3 & 4 unless you want to generate your own X-ray data from FEM scratch.

| Cell | What it does | Required? |
|---|---|---|
| 1. Install | Clone repo + install pinned deps | ✅ always |
| 2. Download data | Pre-computed projections + GT volumes (**uniform compression** default) | ✅ default path |
| 3. FEM + render *(optional)* | Re-run FEM simulation and render projections yourself | ⬜ advanced |
| 4. Stage dataset *(optional)* | Rebuild transforms JSON from your own renders | ⬜ advanced |
| 5. Train | canonical_F → canonical_B → vel_6 → *(vel_9)* → spatiotemporal_mix | ✅ always |
| 6. Visualise | Cross-section comparison + mixing coefficient α(t) | ✅ always |
| 7. TensorBoard | Live training curves | optional |

**Dataset:** Two parallel tracks are available — set `DATASET` in cell 2 & 5:
- `uniform` *(default)* — all surface nodes compressed uniformly
- `indentation` — centre 2×2 patch indented

**Runtime:** T4 GPU. Total time ~45–75 min (install ~10 min, training ~35–60 min). DEMO_FAST skips vel_9 to save ~15 min.

Repo: [NTNU_metamaterials_workshop](https://github.com/igrega348/NTNU_metamaterials_workshop)

## Common errors

**Install failures** — see [`docs/COLAB_DEPENDENCIES.md`](../docs/COLAB_DEPENDENCIES.md). Cell 1 runs `scripts/install_colab_deps.sh` with pinned versions in `requirements-colab.txt`. It must end with **`All imports OK`** and **`numpy: 1.26.4`**. Use **Python 3.12** + **T4** (runtime 2025.10 or 2026.01).

**`ERROR: pip's dependency resolver`** — Colab’s pre-installed packages (tensorflow, jax 0.7, …) conflict on paper; ignore if the script finishes with `All imports OK`.

**`KeyError: 'optimizers'`** — delete partial velocity-field checkpoints and re-run training:

```bash
rm -rf /content/NTNU_metamaterials_workshop/outputs/kelvin_indentation/xray_vfield/
```

In [ ]:
# @title 1. Install dependencies (pinned stack — see docs/COLAB_DEPENDENCIES.md)

import os
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"Python {sys.version_info.major}.{sys.version_info.minor} — need 3.12. "
        "Runtime → Change runtime type → 2025.10 or 2026.01."
    )

%cd /content/

REPO = "/content/NTNU_metamaterials_workshop"
if os.path.isdir(REPO):
    !rm -rf {REPO}
!git clone --recurse-submodules -q https://github.com/igrega348/NTNU_metamaterials_workshop.git

# Use *this* kernel's Python (same as %pip); installs requirements-colab.txt then editables
os.environ["PYTHON"] = sys.executable
!bash {REPO}/scripts/install_colab_deps.sh {REPO}

%load_ext tensorboard
print("Ready:", REPO)

In [ ]:
# @title 2. Download pre-computed data
# DATASET: "uniform" (default) = uniform surface compression; "indentation" = centre patch indented
DATASET = "uniform"  # @param ["uniform", "indentation"]

import os, glob

REPO = "/content/NTNU_metamaterials_workshop"
RELEASE = "https://github.com/igrega348/NTNU_metamaterials_workshop/releases/download/workshop-data-v2"

if DATASET == "uniform":
    tarball = "kelvin_uniform_participant_data.tar.gz"
    data_dir = f"{REPO}/data/kelvin_uniform"
else:
    tarball = "kelvin_indentation_participant_data.tar.gz"
    data_dir = f"{REPO}/data/kelvin_indentation"

os.makedirs(data_dir, exist_ok=True)
!wget -q --show-progress "{RELEASE}/{tarball}" -O /tmp/kelvin_data.tar.gz
!tar -xzf /tmp/kelvin_data.tar.gz -C {REPO}/

imgs = glob.glob(f"{data_dir}/images_*/*.png")
npzs = glob.glob(f"{data_dir}/lattice_*.npz")
print(f"Dataset: {DATASET}  |  Images: {len(imgs)}  |  GT volumes: {len(npzs)}")

In [ ]:
# @title 3. [OPTIONAL] FEM — simulate Kelvin lattice indentation
# Run this cell only if you want to generate your own FEM data instead of using the
# pre-computed projections from cell 2. Skip to cell 5 (Train) for the default path.

SKIP_FEM = True  # ← set False to run FEM from scratch

if not SKIP_FEM:
    import os
    REPO = "/content/NTNU_metamaterials_workshop"
    RUN_NAME = "workshop_colab_custom"   # different name so pre-computed data is not overwritten
    FEM = f"{REPO}/fem_lattice_simulator"
    RUN_DIR = f"{FEM}/runs/{RUN_NAME}"
    os.makedirs(f"{RUN_DIR}/model", exist_ok=True)
    os.makedirs(f"{RUN_DIR}/timesteps", exist_ok=True)
    os.makedirs(f"{RUN_DIR}/yaml", exist_ok=True)

    %cd {FEM}

    # Smaller mesh / fewer steps than desktop run_pipeline.sh (Colab-friendly)
    !python scripts/generate_lattice_from_yaml.py \
      --yaml lattice.yaml --out {RUN_DIR}/model/out.json \
      --nx 4 --ny 4 --nz 4 --subdivide 4

    !python scripts/apply_indent_boundary_conditions.py \
      --in {RUN_DIR}/model/out.json --out {RUN_DIR}/model/out.json \
      --patch-cells-x 2 --patch-cells-y 2 --patch-placement center \
      --indent-uz -0.8 --indenter-uxuy-zero

    # CPU only: Colab jax_cuda12_plugin (0.7.x) conflicts with pinned jaxlib 0.6.2
    !env JAX_PLATFORMS=cpu python -m src.main {RUN_DIR}/model/out.json \
      --ramp-steps 10 --output-prefix {RUN_DIR}/timesteps/{RUN_NAME} --output-every 2

    !python scripts/json_to_yaml.py "{RUN_DIR}/timesteps/{RUN_NAME}_t*.json" \
      --radius-from-area --outdir {RUN_DIR}/yaml --overwrite

    print("FEM YAML timesteps in", f"{RUN_DIR}/yaml")
else:
    print("Skipping FEM (SKIP_FEM=True). Using pre-computed data from cell 2.")

In [ ]:
# @title 4a. [OPTIONAL] Render X-ray projections from your own FEM output
# Only needed if you ran cell 3 with SKIP_FEM=False and want to render your custom geometry.
# Skip to cell 5 (Train) for the default path.

SKIP_RENDER = True  # ← set False if you ran your own FEM in cell 3

if not SKIP_RENDER:
    import glob, os, shutil
    REPO = "/content/NTNU_metamaterials_workshop"
    RUN_NAME = "workshop_colab_custom"
    DATA = f"{REPO}/data/kelvin_indentation"
    YAML_DST = f"{DATA}/yaml"
    YAML_SRC = f"{REPO}/fem_lattice_simulator/runs/{RUN_NAME}/yaml"
    XRAY_DIR = f"{REPO}/neural_xray/xray_projection_render"
    os.makedirs(YAML_DST, exist_ok=True)

    # Download renderer binary + CUDA shared library (v1.7, CUDA 12 for Colab T4)
    RELEASE = "https://github.com/igrega348/xray_projection_render/releases/download/v1.7"
    if not os.path.exists(f"{XRAY_DIR}/xray_render_cuda"):
        !wget -q --show-progress {RELEASE}/xray_projection_render_linux-amd64 \
          -O {XRAY_DIR}/xray_render_cuda
        !chmod +x {XRAY_DIR}/xray_render_cuda
    if not os.path.exists(f"{XRAY_DIR}/libcuda_render.so"):
        !wget -q --show-progress {RELEASE}/libcuda_render-cuda12-linux-x86_64.so \
          -O {XRAY_DIR}/libcuda_render.so
    import os as _os
    _os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + _os.environ.get("LD_LIBRARY_PATH", "")

    for f in glob.glob(f"{YAML_SRC}/*.yaml"):
        shutil.copy2(f, YAML_DST)
    print("Copied", len(glob.glob(f"{YAML_DST}/*.yaml")), "YAML files to", YAML_DST)

    os.environ["YAML_GLOB"] = f"{RUN_NAME}_t*.yaml"
    os.environ["VOLUME_RES"] = "128"
    os.environ["RESOLUTION"] = "256"
    !bash {REPO}/scripts/render_projections.sh

    print("Renders:", glob.glob(f"{DATA}/renders/*/transforms.json"))
else:
    print("Skipping render (SKIP_RENDER=True). Using pre-computed projections from cell 2.")

In [ ]:
# @title 4b. [OPTIONAL] Stage custom renders → transforms JSON
# Only needed if you ran cells 3 & 4a to generate your own renders.
# Skip to cell 5 (Train) for the default path.

SKIP_STAGE = True  # ← set False if you ran your own render in cell 4a

if not SKIP_STAGE:
    REPO = "/content/NTNU_metamaterials_workshop"
    DATA = f"{REPO}/data/kelvin_indentation"
    # lattice_*.npz from renders/*/volume_stage/volume.raw (VOLUME_RES must match render cell)
    !python {REPO}/scripts/stage_kelvin_for_nerf.py \
      --renders-dir {DATA}/renders \
      --out-dir {DATA} \
      --volume-res 128
    print("Staged. transforms files:", sorted(__import__('glob').glob(f"{DATA}/transforms*.json")))
else:
    print("Skipping stage (SKIP_STAGE=True). Using pre-computed transforms from cell 2.")

In [ ]:
# @title 5. Train — canonical_F → canonical_B → vel_6 → spatiotemporal_mix (vel_9 skipped in DEMO_FAST)
# Expected time on T4: ~35–60 min total (DEMO_FAST skips vel_9, saving ~15 min).
# DATASET must match what you chose in cell 2.
# Watch progress below or in TensorBoard (cell 7).

%cd /content
!pip install colab-xterm
%load_ext colabxterm
%env TERM=xterm
from IPython.display import clear_output
import os

DATASET = "uniform"  # @param ["uniform", "indentation"]

clear_output(wait=True)
REPO = "/content/NTNU_metamaterials_workshop"

data_dir = f"{REPO}/data/kelvin_{DATASET}"

if os.path.exists(f"{data_dir}/transforms_00.json"):
    print(
        "\033[1m"
        + "Copy and paste the following command into the terminal window that pops up under this cell."
        + "\033[0m"
    )
    print(
        f"REPO='/content/NTNU_metamaterials_workshop'\n"
        f"DATASET={DATASET} DEMO_FAST=1 ${{REPO}}/scripts/train_kelvin_workshop.sh"
    )
    print()
    print("After training starts, move to TensorBoard visualisation in the cell below")
    print()
    %xterm
else:
    from IPython.core.display import HTML, display
    display(HTML(f'Error: Data not found at {data_dir}/transforms_00.json'))
    display(HTML("Please re-run cell 2 (Download data) with the matching DATASET value."))

In [ ]:
# @title 6. Visualise — cross-sections + mixing coefficient α(t)
# Generates eval_xsections.png (NeRF vs GT density cross-sections at each timestep)
# and eval_mixing.png (spatiotemporal mixing weight α(t)).
# Run after cell 5 completes. DATASET must match cells 2 & 5.

import os
from IPython.display import Image, display

DATASET = "uniform"  # @param ["uniform", "indentation"]

REPO = "/content/NTNU_metamaterials_workshop"
output_dir = f"outputs/kelvin_{DATASET}"
data_dir   = f"data/kelvin_{DATASET}"

!python {REPO}/scripts/eval_kelvin.py \
  --output-dir {REPO}/{output_dir} \
  --data-dir   {REPO}/{data_dir} \
  --timesteps 0 0.2 0.4 0.6 0.8 1.0 \
  --resolution 64 \
  --out-xsec {REPO}/{output_dir}/eval_xsections.png \
  --out-mix  {REPO}/{output_dir}/eval_mixing.png

for fig in [f"{REPO}/{output_dir}/eval_xsections.png", f"{REPO}/{output_dir}/eval_mixing.png"]:
    if os.path.exists(fig):
        print(fig)
        display(Image(fig))
    else:
        print(f"Not found: {fig} — check that training (cell 5) completed")

In [ ]:
# @title 7. TensorBoard — live training curves (run during or after cell 5)
# DATASET must match cells 2, 5 & 6.
DATASET = "uniform"  # @param ["uniform", "indentation"]
REPO = "/content/NTNU_metamaterials_workshop"
%tensorboard --logdir {REPO}/outputs/kelvin_{DATASET}/